# Contextual DPC: 500-seed evaluation with the Atlas eight-output reference

**Ready for local execution: seeds 10001–10500, 300 steps, 200 raw columns, T_ini=2, T_f=10, Clarabel; no SVD.** This is the independent contextual controller with the exact saved K=64, rank-53, p=8 targets; it does not run the Atlas controller. Frozen development settings and provenance are retained; old results were removed from experiment folders and backed up as described below.

Weights are frozen from development tuning: normalized Q=diag(0.01,0.01,0.01,0.01,0.0001,0.0001,100,100), R=0.1, lambda_g_bar=0.1, lambda_sigma=100. The two appended inputs have zero tracking weights. Posture tracking is weak. Each seed uses its exact saved fixed reference from `four_baselines_500_mixed_k_t300_v1/output8/terminal`; no Atlas simulation is launched. Reference loading/construction is excluded from per-step latency and recorded separately.

Run the following **settings**, **preflight**, **run/resume**, and **results** cells. Set `RUN_EVALUATION_500=True` only when ready. Run Select-DPC and contextual DPC sequentially, not simultaneously, for comparable computation timings. Both use separate new output directories. Execution is disabled by default. Previous 100-seed results remain preserved separately; they are not results of this 500-seed test. All statistics use the 300-step horizon (final error windows end at step 300, costs sum steps 1–300, trimmed timing uses steps 6–300).


In [19]:
from pathlib import Path
import os, sys, json, signal, subprocess, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/atlas_deepc/benchmarks/contextual_dpc_terminal_panel.py').is_file())
sys.path.insert(0,str(ROOT/'src'))
from atlas_deepc.benchmarks import contextual_dpc_terminal_panel as evaluation500
# Refresh the launcher after edits, even in a previously used notebook kernel.
evaluation500 = importlib.reload(evaluation500)
assert Path(evaluation500.__file__).resolve() == (ROOT/'src/atlas_deepc/benchmarks/contextual_dpc_terminal_panel.py').resolve(), 'Wrong project imported; restart the kernel'
PYTHON_500 = sys.executable  # Choose the gym-robotics-mujoco kernel.
T_sim = 300
SEED_START_500, SEED_STOP_500, STEPS_500 = 10001, 10500, T_sim
assert evaluation500.SEEDS == list(range(SEED_START_500, SEED_STOP_500 + 1))
assert evaluation500.STEPS == STEPS_500
RESULT_DIR_500 = evaluation500.OUTPUT
RUN_EVALUATION_500 = True
config500, _ = evaluation500.frozen_config()
display(pd.DataFrame([evaluation500.case.asdict(config500)]))
print('500-seed output:',RESULT_DIR_500)
print('Reference source:',evaluation500.ATLAS)


,tini,horizon,columns,warmup,q_xy,r,lambda_g_bar,lambda_sigma,max_iter,tolerance,accept_residual,q_joint,q_dq
0,2,10,200,12,100.0,0.1,0.1,100.0,1000,1.000000e-08,0.0001,0.01,0.0001


500-seed output: /Users/kumiko/Projects/atlas_deepc_reacher/experiments/reacher/contextual_dpc_k64_rank53_terminal8_10001_10500_t300_v1
Reference source: /Users/kumiko/Projects/atlas_deepc_reacher/experiments/reacher/four_baselines_500_mixed_k_t300_v1/output8/terminal


In [20]:
# Read-only validation of all 500 references, offline data and frozen weights.
evaluation500.preflight()
print('Interpreter:',PYTHON_500)
print('Progress:',json.loads((RESULT_DIR_500/'progress.json').read_text())
      if (RESULT_DIR_500/'progress.json').exists() else 'Not started')


Validated 500 fixed K=64, rank-53, eight-output references for seeds 10001--10500, 300 steps.
Frozen config: {'tini': 2, 'horizon': 10, 'columns': 200, 'warmup': 12, 'q_xy': 100.0, 'r': 0.1, 'lambda_g_bar': 0.1, 'lambda_sigma': 100.0, 'max_iter': 1000, 'tolerance': 1e-08, 'accept_residual': 0.0001, 'q_joint': 0.01, 'q_dq': 0.0001}
No controller rollout was run.
Interpreter: /Users/kumiko/miniconda3/envs/gym-robotics-mujoco/bin/python
Progress: Not started


In [ ]:
command500 = [str(PYTHON_500), '-u', '-m', 'atlas_deepc.benchmarks.contextual_dpc_terminal_panel',
              '--output-dir',str(RESULT_DIR_500)]
if RUN_EVALUATION_500:
    env500 = dict(os.environ, PYTHONPATH=str(ROOT/'src'))
    RESULT_DIR_500.mkdir(parents=True,exist_ok=True)
    with (RESULT_DIR_500/'notebook_run.log').open('a',buffering=1) as log500:
        process500 = subprocess.Popen(command500,cwd=ROOT,env=env500,start_new_session=True,
                                      stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process500.stdout:
                print(line,end='',flush=True); log500.write(line)
            returncode500 = process500.wait()
        except KeyboardInterrupt:
            if process500.poll() is None:
                os.killpg(process500.pid,signal.SIGINT)
                try: process500.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process500.pid,signal.SIGTERM)
                    process500.wait(timeout=15)
            print('Stopped. Completed seeds are preserved; the interrupted seed restarts on resume.')
            raise
        if returncode500: raise subprocess.CalledProcessError(returncode500,command500)
else:
    print('Not started. Set RUN_EVALUATION_500=True in the settings cell, then run this cell.')
    print(' '.join(command500))


/Users/kumiko/miniconda3/envs/gym-robotics-mujoco/lib/python3.10/site-packages/cvxpy/reductions/solvers/solving_chain_utils.py:30: UserWarning: The problem includes expressions that don't support CPP backend. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(
1/500 completed, 1 successes; last seed 10001


In [ ]:
# Read-only: load the latest completed-seed summary; no simulation or retuning.
summary500 = RESULT_DIR_500/'summary.json'
if summary500.exists():
    report500 = json.loads(summary500.read_text())
    assert report500['requested'] == 500
    assert all(r['steps'] == STEPS_500 and SEED_START_500 <= r['seed'] <= SEED_STOP_500
               for r in report500['results']), 'Incompatible result panel'
    print(f"Completed {report500['completed']}/500; successes {report500['successes']}/{report500['completed']}")
    print(report500['note'])
    frame500 = pd.DataFrame(report500['results'])
    display(frame500[['seed','strict_success','mean_last_100','max_last_100','fallback_steps']])
    timing500 = [{'module':k,**v} for k,v in report500['modular_timing_steps6_onwards'].items() if v]
    print('Pooled steps 6–300, milliseconds; solve call includes canonicalization/setup.')
    display(pd.DataFrame(timing500))
    if len(frame500):
        fig500,axes500 = plt.subplots(1,3,figsize=(13,4),constrained_layout=True)
        axes500[0].bar(['Success','Failure'],[report500['successes'],len(frame500)-report500['successes']])
        axes500[0].set(title='Completed-seed outcomes',ylabel='Rollouts',ylim=(0,None))
        axes500[1].plot(frame500.seed,1000*frame500.mean_last_100,'.')
        axes500[1].axhline(10,ls='--',color='grey'); axes500[1].set(title='Final-100 mean error',xlabel='Seed',ylabel='mm',ylim=(0,None))
        axes500[2].plot(frame500.seed,[r['timing']['median_ms'] for r in report500['results']],'.')
        axes500[2].set(title='Per-rollout median computation (all steps)',xlabel='Seed',ylabel='ms',ylim=(0,None))
        plt.show()
else:
    print('No completed evaluation summary yet:',summary500)


## Earlier results removed from experiment folders
The old 100-seed, XY-only, and pilot/tuning result folders were cleared at the user's request. 
A verified recovery backup is available at `archive/cleanup_20261018/contextual_old_results_recovery.zip`; the file inventory is in `archive/cleanup_20261018/contextual_old_results_cleanup.json`. 
Frozen tuning settings and provenance remain available. The K=64, rank-53, p=8 reference source is unchanged. 
Old results are not merged into this new evaluation.
